# 01. Data Assembly
**Climate Risk Exposure Index**

Assembles 21 CCKP Excel files into a single clean panel CSV.

**Input:** `cckp_raw/cckp_[country].xlsx` — one file per country, three sheets (`txx`, `rx5day`, `cdd`), wide format with year columns `YYYY-07`.  
**Output:** `cckp_raw/exposure_panel.csv` — 504 rows × 5 columns (`country`, `year`, `temperature`, `precipitation`, `drought`).

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('cckp_raw')
OUTPUT  = RAW_DIR / 'exposure_panel.csv'

CANONICAL_COUNTRIES = [
    'Netherlands', 'United Kingdom', 'France', 'Germany', 'Belgium',
    'Spain', 'Italy', 'Portugal', 'Denmark', 'Sweden', 'Norway',
    'Poland', 'Greece', 'Ireland',
    'Nigeria', 'Kenya', 'South Africa',
    'United States', 'Australia', 'India', 'Brazil',
]

# Sheet name → final column name
SHEET_TO_VARIABLE = {
    'txx':    'temperature',
    'rx5day': 'precipitation',
    'cdd':    'drought',
}

EXPECTED_ROWS = 21 * 24  # 504

# 1. Load in files

In [2]:
def load_country_file(filepath: Path) -> pd.DataFrame:
    """
    Read one CCKP Excel file and return long-format rows:
        country | year | variable | value
    
    Each sheet is wide: columns = ['code', 'name', '1950-07', '1951-07', ...]
    Year columns end in '-07' (CCKP annual ETCCDI reference month).
    Filtered to 2000–2023.
    """
    frames = []
    for sheet, variable in SHEET_TO_VARIABLE.items():
        df = pd.read_excel(filepath, sheet_name=sheet)
        year_cols = [c for c in df.columns if str(c).endswith('-07')]
        long = (
            df[['name'] + year_cols]
            .melt(id_vars='name', var_name='year_str', value_name='value')
        )
        long['year'] = long['year_str'].str[:4].astype(int)
        long = long[long['year'].between(2000, 2023)].copy()
        long['variable'] = variable
        long = long.rename(columns={'name': 'country'})[['country', 'year', 'variable', 'value']]
        frames.append(long)
    return pd.concat(frames, ignore_index=True)


all_frames = []
files = sorted(RAW_DIR.glob('cckp_*.xlsx'))
print(f"Found {len(files)} files\n")

for f in files:
    try:
        df = load_country_file(f)
        all_frames.append(df)
        country = df['country'].iloc[0]
        print(f"  OK   {f.name}  →  {country}  ({len(df)} rows)")
    except Exception as e:
        print(f"  ERR  {f.name}  →  {e}")

long_df = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal rows (long): {len(long_df)}")

Found 21 files

  OK   cckp_australia.xlsx  →  Australia  (72 rows)
  OK   cckp_belgium.xlsx  →  Belgium  (72 rows)
  OK   cckp_brazil.xlsx  →  Brazil  (72 rows)
  OK   cckp_denmark.xlsx  →  Denmark  (72 rows)
  OK   cckp_france.xlsx  →  France  (72 rows)
  OK   cckp_germany.xlsx  →  Germany  (72 rows)
  OK   cckp_greece.xlsx  →  Greece  (72 rows)
  OK   cckp_india.xlsx  →  India  (72 rows)
  OK   cckp_ireland.xlsx  →  Ireland  (72 rows)
  OK   cckp_italy.xlsx  →  Italy  (72 rows)
  OK   cckp_kenya.xlsx  →  Kenya  (72 rows)
  OK   cckp_netherlands.xlsx  →  Netherlands  (72 rows)
  OK   cckp_nigeria.xlsx  →  Nigeria  (72 rows)
  OK   cckp_norway.xlsx  →  Norway  (72 rows)
  OK   cckp_poland.xlsx  →  Poland  (72 rows)
  OK   cckp_portugal.xlsx  →  Portugal  (72 rows)
  OK   cckp_south_africa.xlsx  →  South Africa  (72 rows)
  OK   cckp_spain.xlsx  →  Spain  (72 rows)
  OK   cckp_sweden.xlsx  →  Sweden  (72 rows)
  OK   cckp_united_kingdom.xlsx  →  United Kingdom  (72 rows)
  OK   cckp_un

## 2. Standardise country names 

In [3]:
# Add entries here if CCKP uses different spellings
COUNTRY_MAP = {
    'UK':                       'United Kingdom',
    'Great Britain':            'United Kingdom',
    'USA':                      'United States',
    'United States of America': 'United States',
    'Brasil':                   'Brazil',
    'South Africa RSA':         'South Africa',
}

long_df['country'] = long_df['country'].str.strip().replace(COUNTRY_MAP)

unknown = set(long_df['country'].unique()) - set(CANONICAL_COUNTRIES)
if unknown:
    print("Unrecognised names — add to COUNTRY_MAP:")
    for name in sorted(unknown):
        print(f"  '{name}'")
else:
    print("All country names match canonical list.")

All country names match canonical list.


## 3. Convert to long data format (504 rows × 5 columns)

In [4]:
long_df = long_df[long_df['country'].isin(CANONICAL_COUNTRIES)]

panel = (
    long_df
    .groupby(['country', 'year', 'variable'], as_index=False)['value'].mean()
    .pivot_table(index=['country', 'year'], columns='variable', values='value', aggfunc='mean')
    .reset_index()
)
panel.columns.name = None

# Enforce column order
for col in ['temperature', 'precipitation', 'drought']:
    if col not in panel.columns:
        panel[col] = float('nan')

panel = panel[['country', 'year', 'temperature', 'precipitation', 'drought']]

print(f"Panel shape: {panel.shape}  (expected ({EXPECTED_ROWS}, 5))")
panel.head(10)

Panel shape: (504, 5)  (expected (504, 5))


,country,year,temperature,precipitation,drought
0,Australia,2000,39.43,74.93,88.93
1,Australia,2001,41.26,65.20,76.13
2,Australia,2002,41.03,63.27,135.38
3,Australia,2003,41.38,68.46,78.44
4,Australia,2004,40.75,52.58,87.17
5,Australia,2005,41.48,38.56,86.31
6,Australia,2006,41.50,64.62,87.69
7,Australia,2007,40.43,78.40,84.62
8,Australia,2008,40.26,50.79,88.42
9,Australia,2009,41.18,62.16,105.06


In [5]:
panel.to_csv(OUTPUT, index=False)
print(f"Saved → {OUTPUT}")

Saved → cckp_raw\exposure_panel.csv


---
## 4. Redundancy Check: Is drought distinct from precipitation?
A strong inverse correlation means they double-count the same signal. The result determines whether the final index uses 2 or 3 indicators.

In [6]:
# Cross-sectional summary: one value per country (mean 2000–2023)
cross = panel.groupby('country')[['temperature', 'precipitation', 'drought']].mean()

corr = cross.corr()
print("Correlation matrix (21 countries, 2000–2023 averages):")
print(corr.round(3))

Correlation matrix (21 countries, 2000–2023 averages):
               temperature  precipitation  drought
temperature          1.000          0.461    0.818
precipitation        0.461          1.000    0.513
drought              0.818          0.513    1.000


In [7]:
THRESHOLD = 0.70
r = corr.loc['precipitation', 'drought']
print(f"Precipitation ↔ drought  r = {r:.3f}  (|r| = {abs(r):.3f})")

if abs(r) >= THRESHOLD:
    print(f"\nDECISION: |r| ≥ {THRESHOLD} → drought is REDUNDANT with precipitation.")
    print("Drop drought from the index. Use 2 indicators: temperature + precipitation.")
    print("Document the correlation value as the principled reason in the methodology.")
else:
    print(f"\nDECISION: |r| < {THRESHOLD} → drought adds DISTINCT information.")
    print("Keep all 3 indicators in the final index.")

Precipitation ↔ drought  r = 0.513  (|r| = 0.513)

DECISION: |r| < 0.7 → drought adds DISTINCT information.
Keep all 3 indicators in the final index.


---
## 5. Sanity Checks

In [8]:
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# 1. Row count
flag = 'PASS' if len(panel) == EXPECTED_ROWS else 'FAIL'
print(f"[{flag}] Row count: {len(panel)} (expected {EXPECTED_ROWS})")

# 2. Country coverage per variable
for var in ['temperature', 'precipitation', 'drought']:
    present = set(panel.dropna(subset=[var])['country'].unique())
    missing = set(CANONICAL_COUNTRIES) - present
    flag = 'PASS' if not missing else 'WARN'
    print(f"[{flag}] {var}: {len(present)}/21 countries", end='')
    if missing:
        print(f"  — missing: {sorted(missing)}", end='')
    print()

# 3. Year range
for var in ['temperature', 'precipitation', 'drought']:
    sub = panel.dropna(subset=[var])
    print(f"[INFO] {var}: years {sub['year'].min()}–{sub['year'].max()}")

# 4. Spot-check plausible values
print("\nMean values 2000–2023 (spot-check):")
print(
    panel[panel['country'].isin(['United Kingdom', 'India', 'Nigeria'])]
    .groupby('country')[['temperature', 'precipitation', 'drought']]
    .mean()
    .round(2)
)

# 5. Flood direction: NL, BE, DK should rank high on precipitation
precip_rank = panel.groupby('country')['precipitation'].mean().sort_values(ascending=False)
top5 = precip_rank.head(5).index.tolist()
overlap = {'Netherlands', 'Belgium', 'Denmark'} & set(top5)
flag = 'PASS' if overlap else 'WARN — check precipitation direction'
print(f"\n[{flag}] Top-5 precipitation: {top5}")

# 6. African data gaps
print("\nAfrican data completeness (non-null years per variable):")
for country in ['Nigeria', 'Kenya', 'South Africa']:
    sub = panel[panel['country'] == country]
    for var in ['temperature', 'precipitation', 'drought']:
        n = sub[var].notna().sum()
        print(f"  {country} / {var}: {n}/24")

print("\n" + "=" * 60)
print("Done. Review any WARN or FAIL items above before proceeding.")

SANITY CHECKS
[PASS] Row count: 504 (expected 504)
[PASS] temperature: 21/21 countries
[PASS] precipitation: 21/21 countries
[PASS] drought: 21/21 countries
[INFO] temperature: years 2000–2023
[INFO] precipitation: years 2000–2023
[INFO] drought: years 2000–2023

Mean values 2000–2023 (spot-check):
                temperature  precipitation  drought
country                                            
India                 39.48         127.13    68.80
Nigeria               39.42          85.38    84.05
United Kingdom        25.50          54.19    17.03

[WARN — check precipitation direction] Top-5 precipitation: ['India', 'Brazil', 'Portugal', 'Nigeria', 'Italy']

African data completeness (non-null years per variable):
  Nigeria / temperature: 24/24
  Nigeria / precipitation: 24/24
  Nigeria / drought: 24/24
  Kenya / temperature: 24/24
  Kenya / precipitation: 24/24
  Kenya / drought: 24/24
  South Africa / temperature: 24/24
  South Africa / precipitation: 24/24
  South Africa / dr

In [27]:
data = pd.read_csv("cckp_raw/exposure_panel.csv")
data.head(10)

,country,year,temperature,precipitation,drought
0,Australia,2000,39.43,74.93,88.93
1,Australia,2001,41.26,65.20,76.13
2,Australia,2002,41.03,63.27,135.38
3,Australia,2003,41.38,68.46,78.44
4,Australia,2004,40.75,52.58,87.17
5,Australia,2005,41.48,38.56,86.31
6,Australia,2006,41.50,64.62,87.69
7,Australia,2007,40.43,78.40,84.62
8,Australia,2008,40.26,50.79,88.42
9,Australia,2009,41.18,62.16,105.06
